<a href="https://colab.research.google.com/github/RDGopal/IB9AU-2026/blob/main/FT1_The_Need.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## The Challenge of Financial Sentiment Analysis

This notebook aims to illustrate a critical challenge in natural language processing: the application of general-purpose sentiment analysis models to domain-specific texts, particularly financial news.

The core issue is that **standard sentiment analysis techniques often do not work effectively in financial contexts due to the particular 'financial lingo' and nuances involved.** This notebook will demonstrate this discrepancy by applying a pre-trained, general-purpose BERT-based sentiment model to a financial phrasebank dataset. The expected outcome is a notable disagreement between the model's predictions and the true financial sentiment, thereby highlighting the necessity for specifically fine-tuned models for bespoke financial applications.

In [ ]:
from datasets import load_dataset

ds = load_dataset("warwickai/financial_phrasebank_mirror")

In [ ]:
import pandas as pd

# Convert the 'train' split of the dataset to a pandas DataFrame
df = ds["train"].to_pandas()

# Display the first 5 rows of the DataFrame
display(df.head())

In [ ]:
# Take a random sample of 100 records from the 'train' split of the dataset
# First, shuffle the dataset, then select the first 100 records
ds_sampled = ds["train"].shuffle(seed=42).select(range(100))
df = ds_sampled.to_pandas() # Convert the sampled Dataset object to pandas DataFrame and assign to df

print(df)
print(f"Number of records in the random sample: {len(df)}")

In [ ]:
# Display the first few records of the random sample as a pandas DataFrame
display(df.head())

# Sentiment Analysis
We will perform sentiment analysis on the 'sentence' column of the `df` DataFrame using a pre-trained BERT-based model from Hugging Face, adding 'sentiment' and 'sentiment_score' columns to the DataFrame. Then, visualize the distribution of the predicted sentiments and display the updated DataFrame with the new sentiment columns.

Install the `transformers` library using pip, as it's required for the subsequent steps involving BERT-based models.



In [ ]:
# pip install transformers

Now that the `transformers` library is installed, the next step is to import the `pipeline` function from `transformers` and initialize a sentiment analysis model.



In [ ]:
from transformers import pipeline

# Initialize a sentiment analysis pipeline using a pre-trained BERT-based model
sentiment_analyzer = pipeline('sentiment-analysis', model='nlptown/bert-base-multilingual-uncased-sentiment')

print("Sentiment analyzer loaded successfully.")

With the `sentiment_analyzer` pipeline successfully loaded, the next step is to apply it to the 'sentence' column of the `df` DataFrame to perform sentiment analysis and extract the sentiment label and score.



In [ ]:
sentiments = sentiment_analyzer(list(df['sentence']))

df['sentiment'] = [s['label'] for s in sentiments]
df['sentiment_score'] = [s['score'] for s in sentiments]

print("Sentiment analysis complete. New columns 'sentiment' and 'sentiment_score' added to DataFrame.")

Now that sentiment analysis is complete and the `sentiment` and `sentiment_score` columns have been added to the DataFrame, the next step is to visualize the distribution of the predicted sentiments. A bar plot will effectively show the frequency of each sentiment category.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of predicted sentiments
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='sentiment', order=df['sentiment'].value_counts().index)
plt.title('Distribution of Predicted Sentiments')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()

We display the updated DataFrame, which now includes the `sentiment` and `sentiment_score` columns, to show the results of the sentiment analysis.



In [ ]:
display(df.head())

# Apples-to-Apples
To make a good comparison, we will map the model's 5-star sentiment ratings to 'Negative', 'Neutral', and 'Positive' categories, and do the same for the original numerical labels. Then, calculate agreement metrics, including overall accuracy and a detailed classification report, between the predicted and true sentiment categories. Finally, visualize this agreement with a confusion matrix heatmap and display the updated DataFrame with the new sentiment category columns, along with the calculated agreement percentage and the confusion matrix.

Create a new column `predicted_sentiment_category` by mapping the model's 5-star sentiment ratings ('1 star', '2 stars', '3 stars', '4 stars', '5 stars') to a 3-category scale: 'Negative', 'Neutral', and 'Positive'. Specifically, '1 star' and '2 stars' will map to 'Negative', '3 stars' to 'Neutral', and '4 stars' and '5 stars' to 'Positive'.


In [ ]:
def map_sentiment_to_3_categories(sentiment):
    if sentiment in ['1 star', '2 stars']:
        return 'Negative'
    elif sentiment == '3 stars':
        return 'Neutral'
    elif sentiment in ['4 stars', '5 stars']:
        return 'Positive'
    else:
        return None

df['predicted_sentiment_category'] = df['sentiment'].apply(map_sentiment_to_3_categories)
print("New column 'predicted_sentiment_category' created successfully.")

We now create a new column `true_sentiment_category` by mapping the original numerical labels (0, 1, 2) to the same 3-category sentiment scale: 0 to 'Negative', 1 to 'Neutral', and 2 to 'Positive'.


In [ ]:
def map_label_to_3_categories(label):
    if label == 0:
        return 'Negative'
    elif label == 1:
        return 'Neutral'
    elif label == 2:
        return 'Positive'
    else:
        return None

df['true_sentiment_category'] = df['label'].apply(map_label_to_3_categories)
print("New column 'true_sentiment_category' created successfully.")

Now that both the predicted and true sentiment categories are available, the next step is to calculate the agreement metrics, including overall accuracy and a classification report. This requires importing `accuracy_score` and `classification_report` from `sklearn.metrics`.



In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Calculate overall accuracy
accuracy = accuracy_score(df['true_sentiment_category'], df['predicted_sentiment_category'])
print(f"Overall Agreement (Accuracy): {accuracy:.2f}")

# Generate classification report
report = classification_report(df['true_sentiment_category'], df['predicted_sentiment_category'])
print("\nClassification Report:\n", report)

With the accuracy and classification report already generated, the next step is to visualize the agreement between the predicted and true sentiment categories using a confusion matrix heatmap. This will require `confusion_matrix` from `sklearn.metrics` and `seaborn` for plotting.



In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate the confusion matrix
conf_matrix = confusion_matrix(df['true_sentiment_category'], df['predicted_sentiment_category'])

# Get unique categories for labels
labels = sorted(df['true_sentiment_category'].unique())

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix of Sentiment Categories')
plt.xlabel('Predicted Sentiment Category')
plt.ylabel('True Sentiment Category')
plt.show()

print("Confusion matrix heatmap generated successfully.")

We now display the updated DataFrame with the new sentiment category columns. This will show the `predicted_sentiment_category` and `true_sentiment_category` columns added in previous steps.



In [ ]:
display(df.head())
print("Updated DataFrame with sentiment categories displayed.")

## Summary:

*   The overall agreement (accuracy) between the `true_sentiment_category` and `predicted_sentiment_category` is approximately 0.31.
*   The classification report indicates varying performance across categories:
    *   **Negative sentiment** shows high recall (0.85) but low precision (0.25), suggesting the model correctly identifies most negative cases but also misclassifies many non-negative cases as negative.
    *   **Neutral sentiment** has very low recall (0.12) despite a moderate precision (0.64), indicating the model struggles to identify true neutral sentiments.
    *   **Positive sentiment** demonstrates a balanced performance with a precision of 0.46 and recall of 0.40.
*   The model's overall accuracy of 31% suggests significant room for improvement in sentiment classification. The imbalance in precision and recall across categories, especially for 'Negative' and 'Neutral', indicates that the model is not generalizing well for all sentiment types.

